In [1]:
# mike babb
# find five groups of five letters
# bitmask DFS / backtracking version

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


# DEDUPLICATE MASKS

In [7]:
# multiple words can share the same 5 letters (anagrams), e.g. 'abcde' and
# 'edcba' would both encode to the same bitmask. For the search we only
# care about *which letters* a word occupies, so we search over unique
# masks and expand back out to words afterward via word_byte_to_word_dict.
unique_masks = np.unique(word_byte_array.astype(np.int32))
word_masks = unique_masks.tolist()
print('unique letter-masks:', len(word_masks), 'out of', len(word_byte_array), 'words')

unique letter-masks: 5977 out of 5977 words


# BUILD LETTER -> CANDIDATE WORDS INDEX

In [8]:
# letters_to_words[bit] = every mask that contains that letter (bit 0 = 'a', ... bit 25 = 'z')
letters_to_words = [[] for _ in range(26)]
for m in word_masks:
    for bit in range(26):
        if m & (1 << bit):
            letters_to_words[bit].append(m)

for bit in range(26):
    print(ascii_lowercase[bit], len(letters_to_words[bit]))

a 2545
b 799
c 1154
d 1115
e 2360
f 575
g 838
h 1058
i 1993
j 226
k 831
l 1508
m 968
n 1517
o 1976
p 874
q 91
r 1732
s 1993
t 1418
u 1636
v 405
w 603
x 224
y 1209
z 237


# DFS / BACKTRACKING SEARCH

In [9]:
def find_five_disjoint_groups(letters_to_words):
    """
    Depth-first search for groups of five words whose letters are pairwise
    disjoint (25 unique letters total, since each word is a unique-letter
    5-letter word).

    At each node, instead of trying every remaining word, we look at the
    letters that are still unused and find the one with the FEWEST
    surviving candidate words (candidates that don't collide with letters
    already used). This does two things:

      1. Fails fast: if any unused letter has zero surviving candidates,
         this branch can never produce a valid group, so we prune
         immediately instead of continuing to add words that are doomed
         to backtrack out later.
      2. Keeps branching factor low: we always split on the most
         constrained choice, so the search tree stays narrow. Common
         letters (e, a, r, s, ...) get resolved for free as a side effect
         of other picks rather than being branched on explicitly.

    Because the "next letter to resolve" is chosen deterministically from
    `used_mask` alone (not from the order words were picked), each
    solution is reached along exactly one path - no duplicate solutions
    from re-ordering, and no need for an explicit word-index ordering
    trick.
    """
    solutions = []
    chosen = []

    def dfs(used_mask, depth):
        if depth == 5:
            solutions.append(chosen.copy())
            return

        best_bit, best_candidates = None, None
        for bit in range(26):
            if used_mask & (1 << bit):
                continue
            candidates = [m for m in letters_to_words[bit] if m & used_mask == 0]
            if not candidates:
                return  # this letter can no longer be covered - dead branch
            if best_candidates is None or len(candidates) < len(best_candidates):
                best_bit, best_candidates = bit, candidates

        for m in best_candidates:
            chosen.append(m)
            dfs(used_mask | m, depth + 1)
            chosen.pop()

    dfs(0, 0)
    return solutions

In [10]:
solutions = find_five_disjoint_groups(letters_to_words)
print('solutions found:', len(solutions))

solutions found: 1


# EXPAND MASKS BACK TO WORDS

In [11]:
# each solution is 5 masks; expand every mask back to its actual word(s)
# via word_byte_to_word_dict, since anagrams collapse to the same mask
def expand_solution(mask_group, word_byte_to_word_dict):
    return [word_byte_to_word_dict[m] for m in mask_group]

word_solutions = [expand_solution(sol, word_byte_to_word_dict) for sol in solutions]
word_solutions[:5]

[['hdqrs', 'jowpy', 'vibex', 'muntz', 'flack']]

In [12]:
from itertools import product

# sanity check against the known solution
outcome_set = set(outcome_words)

def solution_contains_outcome(word_solution, outcome_set):
    # each entry in word_solution may itself be a single word or a list of
    # anagram words sharing that mask - normalize before comparing
    for combo in product(*[w if isinstance(w, (list, tuple)) else [w] for w in word_solution]):
        if set(combo) == outcome_set:
            return True
    return False

found = any(solution_contains_outcome(ws, outcome_set) for ws in word_solutions)
print('known solution present:', found)

known solution present: False
